In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Procura a raiz do repositório a partir da pasta notebooks/, da raiz ou de src/.
caminho_atual = Path.cwd().resolve()
raiz = next(
    (
        pasta
        for pasta in (caminho_atual, *caminho_atual.parents)
        if (pasta / "resultados" / "tabelas" / "e3_sessoes.csv").is_file()
    ),
    None,
)

if raiz is None:
    raise FileNotFoundError(
        "Não encontrei resultados/tabelas/e3_sessoes.csv. "
        "Execute o notebook dentro do repositório que já contém os CSVs."
    )

pasta_tabelas = raiz / "resultados" / "tabelas"
arquivo_sessoes = pasta_tabelas / "e3_sessoes.csv"
arquivo_trajetorias = pasta_tabelas / "e3_trajetorias.csv"
arquivo_resumo = pasta_tabelas / "e3_resumo_estatistico.csv"

pasta_graficos = raiz / "resultados" / "graficos" / "notebooks"
pasta_graficos.mkdir(parents=True, exist_ok=True)

for caminho in (arquivo_sessoes, arquivo_trajetorias, arquivo_resumo):
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {caminho}")

print("Raiz:", raiz)
print("Tabelas:", pasta_tabelas)
print("Gráficos serão salvos em:", pasta_graficos)

In [ ]:
df_sessoes = pd.read_csv(arquivo_sessoes)
df_trajetorias = pd.read_csv(arquivo_trajetorias)
df_resumo = pd.read_csv(arquivo_resumo)

# Normaliza o campo booleano para não depender da forma como o CSV foi lido.
df_sessoes["esgotamento"] = (
    df_sessoes["esgotamento"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False})
)
assert df_sessoes["esgotamento"].notna().all(), "Valores inválidos em 'esgotamento'."

print("Sessões individuais (primeiras linhas):")
display(df_sessoes.head())

print("Trajetórias exportadas (primeiras linhas):")
display(df_trajetorias.head())

print("Resumo gerado pelo simulador:")
display(df_resumo)

In [ ]:
colunas_sessoes = {
    "id_participante", "saldo_inicial", "saldo_final",
    "rodadas_executadas", "esgotamento", "classificacao"
}
colunas_trajetorias = {"id_participante", "rodada", "saldo"}

assert colunas_sessoes.issubset(df_sessoes.columns), "Colunas ausentes no CSV de sessões."
assert colunas_trajetorias.issubset(df_trajetorias.columns), "Colunas ausentes no CSV de trajetórias."

assert len(df_sessoes) == 100, "O experimento exige 100 participantes."
assert df_sessoes["id_participante"].is_unique, "IDs de participantes repetidos."
assert set(df_sessoes["id_participante"]) == set(range(1, 101))
assert np.allclose(df_sessoes["saldo_inicial"], 100.0)
assert df_sessoes["rodadas_executadas"].between(1, 500).all()
assert (df_sessoes["saldo_final"] >= 0).all(), "Há saldo final negativo."

# A classificação deve seguir o saldo final, e não o número de rodadas.
classificacao_recalculada = np.select(
    [
        df_sessoes["saldo_final"] > 100,
        df_sessoes["saldo_final"] < 100,
    ],
    ["ganho", "perda"],
    default="empate"
)
assert (df_sessoes["classificacao"].to_numpy() == classificacao_recalculada).all()
assert (
    df_sessoes["esgotamento"].to_numpy()
    == (df_sessoes["saldo_final"].to_numpy() < 1.0)
).all(), "A classificação de esgotamento diverge do saldo final."

# O CSV de trajetórias contém apenas os cinco primeiros jogadores.
ids_primeiros = [1, 2, 3, 4, 5]
assert set(df_trajetorias["id_participante"]) == set(ids_primeiros)

for participante in ids_primeiros:
    traj = (
        df_trajetorias[df_trajetorias["id_participante"] == participante]
        .sort_values("rodada")
    )
    sessao = df_sessoes.set_index("id_participante").loc[participante]

    assert traj["rodada"].tolist() == list(range(501)), (
        f"Trajetória do participante {participante} deve cobrir 0 a 500."
    )
    assert np.isclose(traj.iloc[0]["saldo"], 100.0)
    assert np.isclose(traj.iloc[-1]["saldo"], sessao["saldo_final"])
    assert np.isclose(
        traj.loc[traj["rodada"] == sessao["rodadas_executadas"], "saldo"].iloc[0],
        sessao["saldo_final"]
    )
    saldos_apos_parada = traj.loc[
        traj["rodada"] >= sessao["rodadas_executadas"], "saldo"
    ].to_numpy()
    assert np.allclose(saldos_apos_parada, sessao["saldo_final"]), (
        f"Trajetória do participante {participante} muda após o encerramento."
    )

print("Conferências estruturais e de consistência: OK.")
print("Sessões analisadas:", len(df_sessoes))
print("Pontos das trajetórias:", len(df_trajetorias))

In [ ]:
saldo_inicial = 100.0
saldos = df_sessoes["saldo_final"]

contagens = df_sessoes["classificacao"].value_counts().reindex(
    ["ganho", "empate", "perda"], fill_value=0
)
tabela_categorias = pd.DataFrame({
    "Categoria": ["Ganho", "Empate", "Perda"],
    "Participantes": contagens.to_numpy(),
    "Proporção (%)": (100 * contagens / len(df_sessoes)).to_numpy(),
})

n_esgotamentos = int(df_sessoes["esgotamento"].sum())
taxa_esgotamento = 100 * n_esgotamentos / len(df_sessoes)

estatisticas = pd.Series({
    "Quantidade de participantes": len(df_sessoes),
    "Saldo final médio": saldos.mean(),
    "Saldo final mediano": saldos.median(),
    "Desvio-padrão amostral": saldos.std(ddof=1),
    "Saldo mínimo": saldos.min(),
    "Q1 (25%)": saldos.quantile(0.25),
    "Q3 (75%)": saldos.quantile(0.75),
    "Saldo máximo": saldos.max(),
    "Média de rodadas": df_sessoes["rodadas_executadas"].mean(),
    "Mediana de rodadas": df_sessoes["rodadas_executadas"].median(),
    "Esgotamentos": n_esgotamentos,
    "Taxa de esgotamento (%)": taxa_esgotamento,
})

assert contagens.sum() == 100
assert n_esgotamentos <= int(contagens["perda"])

# Verifica as principais métricas do resumo exportado pelo backend.
resumo_exportado = df_resumo.set_index("metrica")["valor"]
def valor_resumo(rotulo):
    return float(str(resumo_exportado.loc[rotulo]).replace(",", "."))

assert abs(saldos.mean() - valor_resumo("Saldo final médio (créditos)")) < 0.011
assert abs(saldos.median() - valor_resumo("Saldo final mediano (créditos)")) < 0.011
assert abs(saldos.std(ddof=1) - valor_resumo("Saldo final desvio-padrão (créditos)")) < 0.011
# Exemplo de campo CSV: "67 (67.0%)".
texto_esgotamento = str(resumo_exportado.loc["Taxa de Esgotamento (saldo < 1)"])
qtd_exportada = int(texto_esgotamento.split(" (")[0])
assert n_esgotamentos == qtd_exportada, "Quantidade de esgotamentos diverge do resumo."
assert abs(taxa_esgotamento - float(texto_esgotamento.split("(")[1].rstrip("%)"))) < 0.051

print("Categorias finais:")
display(tabela_categorias.style.format({"Proporção (%)": "{:.1f}%"}))
print("Estatísticas recalculadas:")
display(estatisticas.to_frame("Valor").style.format("{:.2f}"))
print("Conferência dos valores exportados: OK.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for participante in ids_primeiros:
    traj = df_trajetorias.loc[
        df_trajetorias["id_participante"] == participante
    ].sort_values("rodada")
    sessao = df_sessoes.set_index("id_participante").loc[participante]

    ax.plot(
        traj["rodada"],
        traj["saldo"],
        linewidth=1.5,
        label=f"Participante {participante} (final: {sessao['saldo_final']:.0f})"
    )

ax.axhline(saldo_inicial, color="black", linestyle="--", linewidth=1.3,
           label="Saldo inicial: 100")
ax.set(
    title="Experimento 3 — Trajetórias dos cinco primeiros participantes",
    xlabel="Rodada",
    ylabel="Saldo (créditos)",
    xlim=(0, 500),
)
ax.grid(alpha=0.25)
ax.legend(fontsize=9)
fig.tight_layout()

saida_trajetorias = pasta_graficos / "e3_trajetorias_5_participantes.png"
fig.savefig(saida_trajetorias, dpi=200, bbox_inches="tight")
plt.show()
print("Gráfico salvo em:", saida_trajetorias)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

# Intervalos regulares de 20 créditos; incluem o saldo zero.
limite_hist = max(400.0, float(np.ceil(saldos.max() / 20) * 20))
bins = np.arange(-10, limite_hist + 21, 20)

ax.hist(saldos, bins=bins, edgecolor="white", linewidth=0.8)
ax.axvline(saldo_inicial, color="black", linestyle="--", linewidth=1.5,
           label="Saldo inicial: 100")
ax.axvline(saldos.mean(), color="tab:orange", linestyle=":", linewidth=1.8,
           label=f"Média final: {saldos.mean():.2f}")
ax.set(
    title="Experimento 3 — Distribuição dos saldos finais (100 participantes)",
    xlabel="Saldo final (créditos)",
    ylabel="Número de participantes",
)
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

saida_histograma = pasta_graficos / "e3_histograma_saldos_finais.png"
fig.savefig(saida_histograma, dpi=200, bbox_inches="tight")
plt.show()
print("Gráfico salvo em:", saida_histograma)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))

ax.boxplot(
    saldos.to_numpy(),
    vert=False,
    widths=0.42,
    showmeans=True,
    meanprops={"marker": "D", "markerfacecolor": "tab:orange",
               "markeredgecolor": "tab:orange"},
)
ax.axvline(saldo_inicial, color="black", linestyle="--", linewidth=1.3,
           label="Saldo inicial: 100")
ax.set(
    title="Experimento 3 — Boxplot dos saldos finais",
    xlabel="Saldo final (créditos)",
    yticks=[1],
    yticklabels=["100 participantes"],
)
ax.legend()
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()

saida_boxplot = pasta_graficos / "e3_boxplot_saldos_finais.png"
fig.savefig(saida_boxplot, dpi=200, bbox_inches="tight")
plt.show()
print("Gráfico salvo em:", saida_boxplot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].bar(tabela_categorias["Categoria"], tabela_categorias["Participantes"])
axes[0].set(title="Resultado final das sessões",
            ylabel="Número de participantes", ylim=(0, 100))
axes[0].grid(axis="y", alpha=0.25)

for i, valor in enumerate(tabela_categorias["Participantes"]):
    axes[0].text(i, valor + 1.5, str(int(valor)), ha="center")

axes[1].bar(["Esgotaram", "Não esgotaram"],
            [n_esgotamentos, len(df_sessoes) - n_esgotamentos])
axes[1].set(title="Esgotamento do saldo (análise separada)",
            ylabel="Número de participantes", ylim=(0, 100))
axes[1].grid(axis="y", alpha=0.25)

for i, valor in enumerate([n_esgotamentos, len(df_sessoes) - n_esgotamentos]):
    axes[1].text(i, valor + 1.5, str(int(valor)), ha="center")

fig.tight_layout()
saida_categorias = pasta_graficos / "e3_categorias_e_esgotamento.png"
fig.savefig(saida_categorias, dpi=200, bbox_inches="tight")
plt.show()
print("Gráfico salvo em:", saida_categorias)

In [ ]:
print("EXPERIMENTO 3 — SÍNTESE")
print(f"Participantes virtuais: {len(df_sessoes)} | Seed: 42")
print("Saldo inicial: 100 créditos | Aposta: 1 crédito | Máximo: 500 rodadas")
print()
print(
    f"Ganho: {int(contagens['ganho'])} ({100*contagens['ganho']/100:.1f}%) | "
    f"Empate: {int(contagens['empate'])} ({100*contagens['empate']/100:.1f}%) | "
    f"Perda: {int(contagens['perda'])} ({100*contagens['perda']/100:.1f}%)"
)
print(f"Esgotamento: {n_esgotamentos} ({taxa_esgotamento:.1f}% dos participantes)")
print(f"Saldo final médio: {saldos.mean():.2f} créditos")
print(f"Saldo final mediano: {saldos.median():.2f} créditos")
print(f"Desvio-padrão amostral: {saldos.std(ddof=1):.2f} créditos")
print(f"Média de rodadas: {df_sessoes['rodadas_executadas'].mean():.1f}")
print()
print("Arquivos exportados pelo notebook:")
for arquivo in [
    saida_trajetorias, saida_histograma, saida_boxplot, saida_categorias
]:
    print(" -", arquivo.relative_to(raiz))